# Music OS — Genuine Qwen3-TTS v8 · Full official interface

Full 1.7B Base voice cloning, CustomVoice preset/style control and VoiceDesign. Optional 0.6B Base/CustomVoice. Choose one model in cell 4 before Run all. Ten supported languages; Hindi is not supported (use Kokoro). Only one full model is loaded per session. Clone only voices with permission. T4 synthesis remains unverified; start with short text. F5 profiles, batch and dialogue remain in notebook 04.


In [ ]:
MODULE = 'qwen'
# Separate Python 3.11 environment; do not install AI packages into Colab's kernel.
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, time, re
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / {'ace': 'ace_step', 'kokoro': 'kokoro', 'f5': 'qwen_voice', 'qwen': 'qwen_voice'}[MODULE]
VOICE_BANK = ROOT / 'voice_bank'
MODELS = ROOT / 'models' / 'ace_step'
for p in (CACHE, OUT, VOICE_BANK, MODELS): p.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV['HF_HOME'] = str(CACHE)
ENV['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
ENV.pop('TRANSFORMERS_CACHE', None)
ENV['PYTHONUNBUFFERED'] = '1'
# Prevent Colab kernel paths from leaking into the isolated interpreter.
ENV.pop('PYTHONPATH', None)
ENV.pop('PYTHONHOME', None)
ENV['PYTHONNOUSERSITE'] = '1'
WORK = Path('/content/ai_music_v8') / MODULE
WORK.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'install_v8.log'
def run_logged(cmd, cwd=None):
    with LOG.open('a') as log:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError(f'Command failed ({result.returncode}). Complete output: {LOG}')
# Colab's system Python can fail at venv/ensurepip. Download the official uv
# wheel with existing pip and extract only its binary, without installing it
# into the kernel or running ensurepip. uv manages the separate AI interpreter.
import zipfile, shutil
UV_HOME = Path('/content/ai_music_v8/bootstrap')
UV_HOME.mkdir(parents=True, exist_ok=True)
UV = UV_HOME / 'uv'
if not UV.exists():
    WHEELS = UV_HOME / 'wheels'
    WHEELS.mkdir(exist_ok=True)
    run_logged([sys.executable, '-m', 'pip', 'download', '--no-cache-dir',
                '--no-deps', '--only-binary=:all:', '--index-url', 'https://pypi.org/simple',
                '--dest', str(WHEELS), 'uv==0.13.8'])
    wheels = sorted(WHEELS.glob('uv-0.13.8-*.whl'))
    if len(wheels) != 1:
        raise RuntimeError(f'Expected one uv wheel, found {len(wheels)}. See {LOG}')
    temporary = UV.with_suffix('.tmp')
    try:
        with zipfile.ZipFile(wheels[0]) as archive:
            binaries = [name for name in archive.namelist() if name.endswith('.data/scripts/uv')]
            if len(binaries) != 1: raise RuntimeError('Official uv wheel has no unique Linux binary.')
            with archive.open(binaries[0]) as src, temporary.open('wb') as dst:
                shutil.copyfileobj(src, dst)
        temporary.chmod(0o755)
        temporary.replace(UV)
    finally:
        temporary.unlink(missing_ok=True)
run_logged([str(UV), '--version'])
run_logged([str(UV), 'python', 'install', '3.11.18'])
VENV = WORK / 'venv'
PY = VENV / 'bin' / 'python'
if not PY.exists(): run_logged([str(UV), 'venv', '--python', '3.11.18', str(VENV)])
print('Colab kernel:', sys.version.split()[0], '| AI runtime: Python 3.11.18')
print('Outputs:', OUT, '| Installation log:', LOG)


In [ ]:
LOCK_CONTENT = '# This file was autogenerated by uv via the following command:\n#    uv pip compile /tmp/music-v8/qwen.in --python-version 3.11 --index-strategy unsafe-best-match -o /tmp/music-v8/AI_Music_OS-main/requirements/qwen.lock\naccelerate==1.12.0\n    # via qwen-tts\naiofiles==23.2.1\n    # via gradio\nannotated-doc==0.0.6\n    # via\n    #   fastapi\n    #   typer\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   fastapi\n    #   gradio\n    #   httpx\n    #   starlette\naudioread==3.1.0\n    # via librosa\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncffi==2.1.1\n    # via soundfile\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.1\n    # via uvicorn\ncloudpickle==3.1.2\n    # via joblib\ndecorator==5.3.1\n    # via librosa\neinops==0.8.2\n    # via qwen-tts\nfastapi==0.162.7\n    # via gradio\nffmpy==1.0.0\n    # via gradio\nfilelock==4.2.10\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nflatbuffers==25.12.19\n    # via onnxruntime\nfsspec==2026.9.0\n    # via\n    #   gradio-client\n    #   huggingface-hub\n    #   torch\ngradio==5.22.0\n    # via\n    #   -r /tmp/music-v8/qwen.in\n    #   qwen-tts\ngradio-client==1.8.0\n    # via gradio\ngroovy==0.1.2\n    # via gradio\nh11==0.16.0\n    # via\n    #   httpcore\n    #   uvicorn\nhf-xet==1.8.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpx==0.28.1\n    # via\n    #   gradio\n    #   gradio-client\n    #   safehttpx\nhuggingface-hub==0.36.2\n    # via\n    #   accelerate\n    #   gradio\n    #   gradio-client\n    #   tokenizers\n    #   transformers\nidna==3.20\n    # via\n    #   anyio\n    #   httpx\n    #   requests\njinja2==3.1.6\n    # via\n    #   gradio\n    #   torch\njoblib==1.6.0\n    # via\n    #   librosa\n    #   scikit-learn\nlazy-loader==0.6\n    # via librosa\nlibrosa==0.11.0\n    # via qwen-tts\nllvmlite==0.50.0\n    # via numba\nmarkdown-it-py==4.3.0\n    # via rich\nmarkupsafe==3.0.4\n    # via\n    #   gradio\n    #   jinja2\nmdurl==0.1.2\n    # via markdown-it-py\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via librosa\nnarwhals==2.28.0\n    # via scikit-learn\nnetworkx==3.6.1\n    # via torch\nnumba==0.68.1\n    # via librosa\nnumpy==1.26.4\n    # via\n    #   -r /tmp/music-v8/qwen.in\n    #   accelerate\n    #   gradio\n    #   librosa\n    #   numba\n    #   onnxruntime\n    #   pandas\n    #   scikit-learn\n    #   scipy\n    #   soundfile\n    #   sox\n    #   soxr\n    #   transformers\nnvidia-cublas-cu12==12.4.5.8\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.4.127\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.4.127\n    # via torch\nnvidia-cuda-runtime-cu12==12.4.127\n    # via torch\nnvidia-cudnn-cu12==9.1.0.70\n    # via torch\nnvidia-cufft-cu12==11.2.1.3\n    # via torch\nnvidia-curand-cu12==10.3.5.147\n    # via torch\nnvidia-cusolver-cu12==11.6.1.9\n    # via torch\nnvidia-cusparse-cu12==12.3.1.170\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.6.2\n    # via torch\nnvidia-nccl-cu12==2.21.5\n    # via torch\nnvidia-nvjitlink-cu12==12.4.127\n    # via\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.4.127\n    # via torch\nonnxruntime==1.32.0\n    # via qwen-tts\nopentelemetry-api==1.45.2\n    # via fastapi\norjson==3.13.0\n    # via gradio\npackaging==26.3\n    # via\n    #   accelerate\n    #   gradio\n    #   gradio-client\n    #   huggingface-hub\n    #   lazy-loader\n    #   onnxruntime\n    #   pooch\n    #   transformers\npandas==2.3.3\n    # via gradio\npillow==11.3.0\n    # via gradio\npip==25.1.1\n    # via -r /tmp/music-v8/qwen.in\nplatformdirs==4.14.5\n    # via pooch\npooch==1.9.0\n    # via librosa\nprotobuf==7.36.2\n    # via onnxruntime\npsutil==7.2.2\n    # via accelerate\npycparser==3.12\n    # via cffi\npydantic==2.14.3\n    # via\n    #   fastapi\n    #   gradio\n    #   typer\npydantic-core==2.50.4\n    # via pydantic\npydub==0.25.1\n    # via gradio\npygments==2.21.0\n    # via rich\npython-dateutil==4.0.0\n    # via pandas\npython-multipart==0.0.33\n    # via gradio\npytz==2026.6\n    # via pandas\npyyaml==6.0.3\n    # via\n    #   accelerate\n    #   gradio\n    #   huggingface-hub\n    #   transformers\nqwen-tts @ https://github.com/QwenLM/Qwen3-TTS/archive/022e286b98fbec7e1e916cb940cdf532cd9f488e.zip\n    # via -r /tmp/music-v8/qwen.in\nregex==2026.11.9\n    # via transformers\nrequests==2.34.2\n    # via\n    #   huggingface-hub\n    #   pooch\n    #   transformers\nrich==15.0.0\n    # via typer\nruff==0.17.5\n    # via gradio\nsafehttpx==0.1.7\n    # via gradio\nsafetensors==0.8.0\n    # via\n    #   accelerate\n    #   transformers\nscikit-learn==1.9.1\n    # via librosa\nscipy==1.17.1\n    # via\n    #   librosa\n    #   scikit-learn\nsemantic-version==2.10.0\n    # via gradio\nshellingham==1.5.4\n    # via typer\nsoundfile==0.14.0\n    # via\n    #   librosa\n    #   qwen-tts\nsox==1.5.0\n    # via qwen-tts\nsoxr==1.1.0\n    # via librosa\nstarlette==0.52.1\n    # via\n    #   fastapi\n    #   gradio\nsympy==1.13.1\n    # via torch\nthreadpoolctl==3.8.0\n    # via scikit-learn\ntokenizers==0.22.2\n    # via transformers\ntomlkit==0.13.3\n    # via gradio\ntorch==2.6.0+cu124\n    # via\n    #   -r /tmp/music-v8/qwen.in\n    #   accelerate\n    #   torchaudio\ntorchaudio==2.6.0+cu124\n    # via\n    #   -r /tmp/music-v8/qwen.in\n    #   qwen-tts\ntqdm==4.70.1\n    # via\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.3\n    # via qwen-tts\ntriton==3.2.0\n    # via torch\ntyper==0.29.1\n    # via gradio\ntyping-extensions==4.16.0\n    # via\n    #   anyio\n    #   fastapi\n    #   gradio\n    #   gradio-client\n    #   huggingface-hub\n    #   librosa\n    #   opentelemetry-api\n    #   pydantic\n    #   pydantic-core\n    #   soundfile\n    #   sox\n    #   starlette\n    #   torch\n    #   typing-inspection\ntyping-inspection==0.4.5\n    # via\n    #   fastapi\n    #   pydantic\ntzdata==2026.6\n    # via\n    #   pandas\n    #   python-dateutil\nurllib3==2.8.0\n    # via requests\nuvicorn==0.54.0\n    # via gradio\nwebsockets==15.0.1\n    # via gradio-client\n'
CHECK_IMPORTS = 'import torch, torchaudio; from qwen_tts import Qwen3TTSModel; from qwen_tts.cli.demo import build_demo; print("Qwen imports OK")'
run_logged(['apt-get', '-qq', 'update'])
run_logged(['apt-get', '-qq', 'install', '-y', 'sox', 'ffmpeg', 'libsndfile1'])
LOCK = WORK / 'requirements.lock'
LOCK.write_text(LOCK_CONTENT)
run_logged([str(UV), 'pip', 'sync', '--python', str(PY), '--no-cache', '--index', 'https://download.pytorch.org/whl/cu124', '--index', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match', str(LOCK)])
run_logged([str(UV), 'pip', 'check', '--python', str(PY)])
run_logged([str(PY), '-c', CHECK_IMPORTS])
print('Qwen installation + imports: OK')


In [ ]:
# Verify the actual AI runtime, not Colab's preinstalled torch.
run_logged([str(PY), '-c', "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type, then rerun.'"])
print('T4/CUDA check: OK')


In [ ]:
# Choose one FULL official checkpoint. Changing it requires rerunning this and launch cell.
QWEN_MODEL = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base' #@param ['Qwen/Qwen3-TTS-12Hz-1.7B-Base', 'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice', 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', 'Qwen/Qwen3-TTS-12Hz-0.6B-Base', 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice']
ENV['AI_MUSIC_QWEN_MODEL'] = QWEN_MODEL
ENV['AI_MUSIC_OUT'] = str(OUT)
ENV['AI_MUSIC_VOICE_BANK'] = str(VOICE_BANK)
# Generated Gradio files persist under the existing output folder.
ENV['GRADIO_TEMP_DIR'] = str(OUT / 'qwen_generated')
(WORK / 'model_config.py').write_text('"""Full official model options and conservative Colab launch settings."""\nQWEN_MODELS = [\'Qwen/Qwen3-TTS-12Hz-1.7B-Base\', \'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\', \'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign\', \'Qwen/Qwen3-TTS-12Hz-0.6B-Base\', \'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice\']\nACE_MODELS = [\'acestep-v15-turbo\', \'acestep-v15-sft\', \'acestep-v15-base\', \'acestep-v15-xl-turbo\', \'acestep-v15-xl-sft\', \'acestep-v15-xl-base\']\nLM_MODELS = [\'None\', \'acestep-5Hz-lm-0.6B\', \'acestep-5Hz-lm-1.7B\', \'acestep-5Hz-lm-4B\']\n\n\ndef ace_command(exe, model, lm, outputs):\n    if model not in ACE_MODELS or lm not in LM_MODELS:\n        raise ValueError(\'Unsupported ACE-Step model.\')\n    cmd = [str(exe), \'--port\', \'7865\', \'--server-name\', \'0.0.0.0\', \'--share\', \'--service_mode\', \'false\', \'--init_service\', \'true\', \'--config_path\', model, \'--device\', \'cuda\', \'--backend\', \'pt\', \'--init_llm\', \'false\' if lm == \'None\' else \'true\', \'--use_flash_attention\', \'false\', \'--offload_to_cpu\', \'true\', \'--offload_dit_to_cpu\', \'true\', \'--quantization\', \'int8_weight_only\', \'--batch_size\', \'1\', \'--allowed-path\', str(outputs), \'--download-source\', \'huggingface\']\n    if lm != \'None\': cmd += [\'--lm_model_path\', lm]\n    return cmd\n\n\ndef qwen_settings(model_id):\n    if model_id not in QWEN_MODELS:\n        raise ValueError(\'Unsupported Qwen3-TTS model.\')\n    return dict(device_map=\'cuda:0\', dtype=\'float16\', attn_implementation=\'sdpa\')\n', encoding='utf-8')
(WORK / 'qwen_app.py').write_text('"""Full official Qwen3-TTS demo with T4-compatible loading and Drive exports."""\nimport os\nfrom pathlib import Path\nfrom model_config import qwen_settings\n\n\ndef main():\n    import torch\n    from qwen_tts import Qwen3TTSModel\n    from qwen_tts.cli.demo import build_demo\n    model_id = os.environ[\'AI_MUSIC_QWEN_MODEL\']\n    settings = qwen_settings(model_id)\n    settings[\'dtype\'] = torch.float16\n    out = Path(os.environ[\'AI_MUSIC_OUT\'])\n    out.mkdir(parents=True, exist_ok=True)\n    print(\'Loading full official model:\', model_id, flush=True)\n    model = Qwen3TTSModel.from_pretrained(model_id, **settings)\n    demo = build_demo(model, model_id, {\'max_new_tokens\': 2048})\n    demo.queue(default_concurrency_limit=1).launch(share=True, server_name=\'0.0.0.0\', server_port=7867, allowed_paths=[str(out)])\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(WORK / 'audio_tools.py').write_text('"""Shared local audio exports; embedded beside each standalone notebook app."""\nfrom pathlib import Path\nimport json\nimport math\nimport subprocess\nimport time\nimport zipfile\nimport numpy as np\nimport soundfile as sf\n\n\ndef read_text_file(path):\n    if not path:\n        raise ValueError(\'Choose a UTF-8 .txt file.\')\n    p = Path(path)\n    if p.suffix.lower() != \'.txt\' or p.stat().st_size > 1_000_000:\n        raise ValueError(\'Use a .txt file smaller than 1 MB.\')\n    return p.read_text(encoding=\'utf-8-sig\')\n\n\ndef split_text(text, limit=180):\n    """Bound chunks even for punctuation-free text (Espeak otherwise truncates)."""\n    import re\n    chunks = []\n    for line in (text or \'\').strip().splitlines():\n        for part in re.split(r\'(?<=[.!?。！？।])\\s*\', line.strip()):\n            while len(part) > limit:\n                cut = part.rfind(\' \', 0, limit + 1)\n                cut = cut if cut > 0 else limit\n                chunks.append(part[:cut].strip())\n                part = part[cut:].strip()\n            if part:\n                chunks.append(part)\n    return chunks\n\n\ndef export_audio(wav, format=\'WAV\', pitch=0, normalize=False):\n    if format not in (\'WAV\', \'MP3\', \'FLAC\'):\n        raise ValueError(\'Choose WAV, MP3 or FLAC.\')\n    if not math.isfinite(float(pitch)) or not -6 <= float(pitch) <= 6:\n        raise ValueError(\'Pitch must be between -6 and +6 semitones.\')\n    wav = Path(wav)\n    if format == \'WAV\' and not pitch and not normalize:\n        return str(wav)\n    target = wav.with_name(wav.stem + \'_export.\' + format.lower())\n    filters = []\n    if pitch:\n        ratio = 2 ** (float(pitch) / 12)\n        filters += [f\'asetrate={sf.info(wav).samplerate}*{ratio}\', \'aresample=24000\', f\'atempo={1 / ratio}\']\n    if normalize:\n        filters.append(\'loudnorm=I=-16:TP=-1.5:LRA=11\')\n    cmd = [\'ffmpeg\', \'-y\', \'-v\', \'error\', \'-i\', str(wav)]\n    if filters:\n        cmd += [\'-af\', \',\'.join(filters)]\n    if format == \'MP3\':\n        cmd += [\'-codec:a\', \'libmp3lame\', \'-b:a\', \'192k\']\n    subprocess.run(cmd + [str(target)], check=True)\n    return str(target)\n\n\ndef save_metadata(wav, data):\n    Path(wav).with_suffix(\'.json\').write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef package_files(files, out, prefix):\n    path = Path(out) / f\'{prefix}_{time.time_ns()}.zip\'\n    with zipfile.ZipFile(path, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for f in files:\n            p = Path(f)\n            archive.write(p, p.name)\n            for suffix in (\'.json\', \'.srt\'):\n                sidecar = p.with_suffix(suffix)\n                if sidecar.exists():\n                    archive.write(sidecar, sidecar.name)\n    return str(path)\n\n\ndef recent_outputs(out):\n    files = sorted((p for p in Path(out).iterdir() if p.suffix.lower() in (\'.wav\', \'.mp3\', \'.flac\', \'.zip\')), key=lambda p: p.stat().st_mtime, reverse=True)[:30]\n    return [str(p) for p in files]\n\n\ndef timestamp(seconds):\n    ms = round(seconds * 1000)\n    hours, ms = divmod(ms, 3600000)\n    minutes, ms = divmod(ms, 60000)\n    sec, ms = divmod(ms, 1000)\n    return f\'{hours:02}:{minutes:02}:{sec:02},{ms:03}\'\n', encoding='utf-8')
(WORK / 'f5_app.py').write_text("from pathlib import Path\nimport os\nimport time\nimport subprocess\nimport numpy as np\nimport soundfile as sf\nimport gradio as gr\nfrom audio_tools import export_audio, save_metadata, package_files, read_text_file, recent_outputs\nOUT = Path(os.environ['AI_MUSIC_OUT'])\nVOICE_BANK = Path(os.environ['AI_MUSIC_VOICE_BANK'])\nOUT.mkdir(parents=True, exist_ok=True)\nVOICE_BANK.mkdir(parents=True, exist_ok=True)\n_tts = None\n\n\ndef get_tts():\n    global _tts\n    if _tts is None:\n        from f5_tts.api import F5TTS\n        print('Loading F5-TTS (first run downloads model to Drive)...')\n        _tts = F5TTS()\n    return _tts\n\n\ndef profiles():\n    return sorted(p.stem for p in VOICE_BANK.glob('*.wav'))\n\n\ndef profile_paths(name):\n    if not name or name not in profiles():\n        raise gr.Error('Choose a saved voice profile.')\n    return VOICE_BANK / f'{name}.wav', VOICE_BANK / f'{name}.txt'\n\n\ndef load_profile(name):\n    ref, txt = profile_paths(name)\n    return str(ref), txt.read_text(encoding='utf-8') if txt.exists() else ''\n\n\ndef save_voice(sample, profile_name, ref_text, start=0, duration=15, normalize=False, overwrite=False):\n    if sample is None:\n        raise gr.Error('Upload or record a voice sample first.')\n    safe = ''.join(c for c in (profile_name or '') if c.isalnum() or c in (' ', '_', '-')).strip()\n    if not safe:\n        raise gr.Error('Enter a profile name.')\n    if not 0 <= float(start) or not 3 <= float(duration) <= 15:\n        raise gr.Error('Choose a nonnegative start and 3–15 seconds of reference speech.')\n    dst = VOICE_BANK / f'{safe}.wav'\n    if dst.exists() and not overwrite:\n        raise gr.Error('Profile exists; enable overwrite or choose another name.')\n    temp = VOICE_BANK / f'_reference_{time.time_ns()}.wav'\n    cmd = ['ffmpeg', '-y', '-v', 'error', '-ss', str(start), '-i', str(sample), '-t', str(duration), '-ac', '1', '-ar', '24000']\n    if normalize:\n        cmd += ['-af', 'loudnorm=I=-20:TP=-2:LRA=11']\n    try:\n        subprocess.run(cmd + [str(temp)], check=True)\n        audio, sr = sf.read(temp)\n        if len(audio) < 3 * sr or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 0.0001:\n            raise gr.Error('Selected reference must contain at least 3 seconds of audible speech.')\n        temp.replace(dst)\n    finally:\n        temp.unlink(missing_ok=True)\n    (VOICE_BANK / f'{safe}.txt').write_text((ref_text or '').strip(), encoding='utf-8')\n    return f'Saved: {safe}', gr.update(choices=profiles(), value=safe)\n\n\ndef update_transcript(name, text):\n    _, path = profile_paths(name)\n    path.write_text((text or '').strip(), encoding='utf-8')\n    return 'Transcript saved.'\n\n\ndef generate(text, profile_name, speed=1.0, steps=32, seed=-1, cfg=2.0, crossfade=0.15, remove_silence=False, format='WAV', pitch=0, normalize=False):\n    if not (text or '').strip():\n        raise gr.Error('Enter text.')\n    ref, ref_txt = profile_paths(profile_name)\n    if not 0.5 <= float(speed) <= 2 or int(steps) not in (16, 32, 64):\n        raise gr.Error('Speed must be 0.5–2; choose 16, 32 or 64 sampling steps.')\n    if float(seed) != int(seed) or int(seed) < -1 or int(seed) > 2147483647:\n        raise gr.Error('Seed must be -1 (random) or an integer from 0 to 2147483647.')\n    if not 0 <= float(cfg) <= 5 or not 0 <= float(crossfade) <= 0.5:\n        raise gr.Error('Guidance must be 0–5; crossfade must be 0–0.5 seconds.')\n    ref_text = ref_txt.read_text(encoding='utf-8') if ref_txt.exists() else ''\n    out = OUT / f'{profile_name}_{time.time_ns()}.wav'\n    tts = get_tts()\n    tts.infer(ref_file=str(ref), ref_text=ref_text, gen_text=text.strip(), file_wave=str(out), speed=float(speed), nfe_step=int(steps), seed=None if int(seed) == -1 else int(seed), cfg_strength=float(cfg), cross_fade_duration=float(crossfade), remove_silence=bool(remove_silence))\n    if not out.exists() or sf.info(out).frames == 0:\n        raise RuntimeError('F5-TTS did not save audio.')\n    result = export_audio(out, format, pitch, normalize)\n    save_metadata(result, dict(text=text, profile=profile_name, speed=speed, steps=steps, seed=getattr(tts, 'seed', seed), cfg=cfg, crossfade=crossfade, remove_silence=remove_silence, pitch=pitch, normalize=normalize))\n    return result\n\n\ndef ui_generate(*args):\n    try:\n        result = generate(*args)\n        return result, result, Path(result).with_suffix('.json').read_text(encoding='utf-8')\n    except Exception as e:\n        raise gr.Error(str(e)) from e\n\n\ndef batch_generate(text, profile_name, *settings):\n    lines = [line.strip() for line in (text or '').splitlines() if line.strip()]\n    if not lines or len(lines) > 50:\n        raise gr.Error('Enter 1–50 nonempty lines, one clip per line.')\n    try:\n        return package_files([generate(line, profile_name, *settings) for line in lines], OUT, 'f5_batch')\n    except Exception as e:\n        raise gr.Error(str(e)) from e\n\n\ndef dialogue_generate(script, speed=1):\n    lines = [line.strip() for line in (script or '').splitlines() if line.strip()]\n    if not lines or len(lines) > 50:\n        raise gr.Error('Enter 1–50 lines as Profile name: words to speak.')\n    turns = []\n    for line in lines:\n        name, sep, words = line.partition(':')\n        if not sep or not words.strip():\n            raise gr.Error('Every dialogue line must be Profile name: words to speak.')\n        profile_paths(name.strip())\n        turns.append((name.strip(), words.strip()))\n    try:\n        parts, sr = [], None\n        for name, words in turns:\n            path = generate(words, name, speed)\n            audio, rate = sf.read(path)\n            if sr is not None and sr != rate:\n                raise RuntimeError('Dialogue sample rates do not match.')\n            sr = rate\n            if parts:\n                parts.append(np.zeros(round(rate * 0.25)))\n            parts.append(audio)\n        out = OUT / f'dialogue_{time.time_ns()}.wav'\n        sf.write(out, np.concatenate(parts), sr)\n        save_metadata(out, dict(script=script, speed=speed, pause=0.25))\n        return str(out), str(out)\n    except Exception as e:\n        raise gr.Error(str(e)) from e\n\n\ndef build_demo():\n    with gr.Blocks(title='AI Music OS — F5-TTS v7') as demo:\n        gr.Markdown('## F5-TTS Voice Clone · English / Chinese\\nClone only voices you have permission to use.')\n        pick = gr.Dropdown(profiles(), label='Voice profile')\n        refresh = gr.Button('Refresh profiles')\n        with gr.Tab('Save a voice'):\n            sample = gr.Audio(sources=['upload', 'microphone'], type='filepath', label='Reference speech')\n            ref_text = gr.Textbox(label='Reference transcript (blank = automatic transcription)')\n            name = gr.Textbox(label='Profile name')\n            start = gr.Number(value=0, minimum=0, label='Reference start (seconds)')\n            duration = gr.Slider(3, 15, value=15, step=0.5, label='Reference length (seconds)')\n            ref_norm = gr.Checkbox(label='Normalize reference loudness')\n            overwrite = gr.Checkbox(label='Overwrite existing profile', value=False)\n            save = gr.Button('Save profile', variant='primary')\n            status = gr.Textbox(label='Status')\n        with gr.Tab('Preview & edit profile'):\n            preview = gr.Audio(type='filepath', label='Saved reference')\n            transcript = gr.Textbox(lines=4, label='Saved transcript')\n            edit = gr.Button('Save transcript')\n            edit_status = gr.Textbox(label='Status')\n        with gr.Tab('Speak & batch'):\n            text = gr.Textbox(lines=8, label='Text to speak')\n            upload = gr.File(file_types=['.txt'], type='filepath', label='Import text')\n            speed = gr.Slider(0.5, 2, value=1, step=0.05, label='Speed')\n            with gr.Accordion('Quality & export', open=False):\n                steps = gr.Radio([16, 32, 64], value=32, label='Sampling steps')\n                seed = gr.Number(value=-1, precision=0, label='Seed (-1 = random)')\n                cfg = gr.Slider(0, 5, value=2, step=0.1, label='Guidance')\n                crossfade = gr.Slider(0, 0.5, value=0.15, step=0.01, label='Chunk crossfade (seconds)')\n                silence = gr.Checkbox(label='Remove generated silence')\n                format = gr.Radio(['WAV', 'MP3', 'FLAC'], value='WAV', label='Export')\n                pitch = gr.Slider(-6, 6, value=0, step=0.5, label='Pitch shift (semitones)')\n                normalize = gr.Checkbox(label='Normalize output loudness')\n            settings = [speed, steps, seed, cfg, crossfade, silence, format, pitch, normalize]\n            gen = gr.Button('Generate', variant='primary')\n            audio = gr.Audio(type='filepath', label='Cloned speech')\n            download = gr.File(label='Download audio')\n            metadata = gr.Textbox(lines=4, label='Generation settings & seed')\n            batch = gr.Textbox(lines=6, label='Batch: one clip per line')\n            batch_button = gr.Button('Generate batch')\n            archive = gr.File(label='Batch ZIP')\n        with gr.Tab('Multi-voice dialogue'):\n            script = gr.Textbox(lines=10, label='Profile name: words to speak (one turn per line)')\n            dialogue_speed = gr.Slider(0.5, 2, value=1, step=0.05, label='Dialogue speed')\n            dialogue_button = gr.Button('Generate dialogue', variant='primary')\n            dialogue_audio = gr.Audio(type='filepath', label='Dialogue')\n            dialogue_download = gr.File(label='Download dialogue')\n        with gr.Tab('Saved outputs'):\n            output_refresh = gr.Button('Refresh outputs')\n            history = gr.Files(label='Recent downloads')\n        refresh.click(lambda: gr.update(choices=profiles()), outputs=pick)\n        save.click(save_voice, [sample, name, ref_text, start, duration, ref_norm, overwrite], [status, pick])\n        pick.change(load_profile, pick, [preview, transcript])\n        edit.click(update_transcript, [pick, transcript], edit_status)\n        upload.change(read_text_file, upload, text)\n        gen.click(ui_generate, [text, pick] + settings, [audio, download, metadata])\n        batch_button.click(batch_generate, [batch, pick] + settings, archive)\n        dialogue_button.click(dialogue_generate, [script, dialogue_speed], [dialogue_audio, dialogue_download])\n        output_refresh.click(lambda: recent_outputs(OUT), outputs=history)\n    return demo\n\n\nif __name__ == '__main__':\n    build_demo().queue(default_concurrency_limit=1).launch(share=True, server_name='0.0.0.0', server_port=7867, allowed_paths=[str(OUT), str(VOICE_BANK)])\n", encoding='utf-8')
COMMAND = [str(PY), str(WORK / 'qwen_app.py')]


In [ ]:
# Launch in the isolated environment; rerunning safely replaces the previous process.
if 'proc' in globals() and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
LAUNCH_LOG = OUT / 'launch_v8.log'
with LAUNCH_LOG.open('w') as log:
    proc = subprocess.Popen(COMMAND, cwd=str(OUT), env=ENV, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):
    time.sleep(2)
    text = LAUNCH_LOG.read_text(errors='replace')
    if proc.poll() is not None:
        print(text[-18000:])
        raise RuntimeError(f'Application exited. Full log: {LAUNCH_LOG}')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match: url = match.group(0); break
if url is None:
    print(LAUNCH_LOG.read_text(errors='replace')[-6000:])
    raise TimeoutError(f'No share URL after 30 minutes; inspect {LAUNCH_LOG}. Rerun this cell to retry.')
print('Open Music OS:', url)
print('Keep this Colab session open. Logs:', LAUNCH_LOG)


In [ ]:
# Inspect the application log at any time.
print(LAUNCH_LOG.read_text(errors='replace')[-15000:])
